In [1]:
from datasets import load_dataset
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split

In [2]:
REPO_ID = "cais/mmlu"
STORE = Path("../store/mmlu")
STORE.mkdir(parents=True, exist_ok=True)

SUBJECTS = [
    "abstract_algebra",
    "astronomy",
    "anatomy",
    "computer_security",
    "electrical_engineering",
    "marketing",
    "international_law",
    "prehistory",
    "world_religions",
    "high_school_geography",
]

In [3]:
# MMLU has no train split (only test/validation/dev, all small-to-medium).
# Combine all three per subject, then 80/20 split per-concept-balanced.
#   subject  -> concept
#   question -> question (text only, no MCQ options)
#   answer   -> the correct choice text

parts = []
for subject in SUBJECTS:
    for split in ("test", "validation", "dev"):
        ds = load_dataset(REPO_ID, subject, split=split)
        df = pd.DataFrame(ds)
        parts.append(pd.DataFrame({
            "concept":  df["subject"],
            "question": df["question"],
            "answer":   df.apply(lambda r: r["choices"][r["answer"]], axis=1),
        }))
df = pd.concat(parts, ignore_index=True)

for col in ("concept", "question", "answer"):
    df[col] = df[col].fillna("").astype(str).str.strip()
df = df[(df["question"] != "") & (df["answer"] != "")].drop_duplicates(subset=["concept", "question"]).reset_index(drop=True)

min_n = df.groupby("concept").size().min()
balanced = pd.concat(g.sample(min_n, random_state=42) for _, g in df.groupby("concept")).reset_index(drop=True)
train_df, test_df = train_test_split(balanced, test_size=0.2, random_state=42, stratify=balanced["concept"])

train_df.to_csv(STORE / "train.csv", index=False)
test_df.to_csv(STORE / "test.csv", index=False)

def stats(df):
    by = df.groupby("concept").size()
    return {"rows": len(df), "concepts": by.shape[0], "rows/concept (mean)": round(by.mean(), 1), "min-max": f"{by.min()}-{by.max()}"}

print(pd.DataFrame({"train": stats(train_df), "test": stats(test_df)}).T)
print(f"\ntrain/test ratio: {len(train_df)/len(test_df):.2f}")

      rows concepts rows/concept (mean) min-max
train  928       10                92.8   92-93
test   232       10                23.2   23-24

train/test ratio: 4.00


In [4]:
print("concepts:")
for c in sorted(train_df["concept"].unique()):
    print(" ", c)

concepts:
  abstract_algebra
  anatomy
  astronomy
  computer_security
  electrical_engineering
  high_school_geography
  international_law
  marketing
  prehistory
  world_religions
